# Exploration and Exploitation

1. Introduction
2. Multi-Armed Bandits
3. Contextual Bandits
4. MDPs


## Exploration vs. Exploitation Dilemma

- Online decision-making involves a fundamental choice:
    - **Exploitation** Make the best decision given current information
    - **Exploration** Gather more information
- The best long-term strategy may involve short-term sacrifices
- Gather enough information to make the best overall decisions

### Examples:

- Restaurant Selection
    - Exploitation Go to your favourite restaurant
    - Exploration Try a new restaurant
- Online Banner Advertisements
    - Exploitation Show the most successful advert
    - Exploration Show a diﬀerent advert
- Oil Drilling
    - Exploitation Drill at the best known location
    - Exploration Drill at a new location
- Game Playing
    - Exploitation Play the move you believe is best
    - Exploration Play an experimental move


### Principles

#### Naive Exploration
- Add noise to greedy policy (e.g. ε-greedy)
- The simplest approach: occasionally pick a random action instead of the best known one, so the agent stumbles upon unexplored options.

#### Optimistic Initialisation
- Assume the best until proven otherwise
- Start with inflated value estimates for all actions. The agent will naturally try under-explored actions first because they still look promising on paper.

#### Optimism in the Face of Uncertainty
- Prefer actions with uncertain values
- Actively seek out actions where the value estimate has high uncertainty (e.g. UCB). Uncertainty is treated as a bonus — unknown = potentially great.

#### Probability Matching
- Select actions according to the probability they are best
- Sample an action proportional to how likely it is to be optimal (e.g. Thompson Sampling). Balances exploration and exploitation probabilistically.

#### Information State Search
- Lookahead search incorporating value of information
- Plan ahead by treating the *knowledge gained* from an action as part of its reward. Explicitly reasons about which actions are most informative.

## The Multi-Armed Bandit

- A multi-armed bandit is a tuple $\langle \mathcal{A}, \mathcal{R} \rangle$
- $\mathcal{A}$ is a known set of $m$ actions (or "arms")
- $\mathcal{R}^a(r) = \mathbb{P}[r|a]$ is an unknown probability distribution over rewards
- At each step $t$ the agent selects an action $a_t \in \mathcal{A}$
- The environment generates a reward $r_t \sim \mathcal{R}^{a_t}$
- The goal is to maximise cumulative reward $\sum_{\tau=1}^{t} r_\tau$

<img src="imgs/image-124.png" width=200px>

### Simple Explanation

Think of a row of slot machines ("one-armed bandits"), each with a different — and unknown — payout rate.

- **Arms**: the $m$ machines you can pull.
- **$\mathcal{R}^a$**: each machine has its own hidden reward distribution; you don't know which is best.
- **At each step**: you pick one machine to pull.
- **Reward**: you observe a random payout sampled from that machine's distribution.
- **Goal**: maximise total winnings over time.

The core challenge is the **explore/exploit trade-off** — do you keep pulling the machine that has paid out best so far, or try others that might be even better?

### Regret

- The **action-value** is the mean reward for action $a$:
$$Q(a) = \mathbb{E}[r|a]$$

- The **optimal value** $V^*$ is:
$$V^* = Q(a^*) = \max_{a \in \mathcal{A}} Q(a)$$

- The **regret** is the opportunity loss for **one step**:
$$l_t = \mathbb{E}[V^* - Q(a_t)]$$

- The **total regret** is the total opportunity loss:
$$L_t = \mathbb{E}\left[\sum_{\tau=1}^{t} V^* - Q(a_\tau)\right]$$

- Maximise cumulative reward $\equiv$ minimise total regret

#### Simple Explanation

- **$Q(a)$**: the average reward you expect from pulling arm $a$.
- **$V^*$**: the average reward of the *best possible* arm — the ceiling you're aiming for.
- **Regret at step $t$**: how much worse your chosen action $a_t$ is compared to always picking the best arm. Zero regret means you picked optimally.
- **Total regret $L_t$**: the accumulated gap over all steps — how much reward you *missed out on* by not always picking the best arm.

Minimising total regret and maximising total reward are exactly the same objective — you want your choices to be as close to the optimal arm as possible, as often as possible.

### Counting Regret

- The **count** $N_t(a)$ is the expected number of selections for action $a$
- The **gap** $\Delta_a$ is the difference in value between action $a$ and optimal action $a^*$:
$$\Delta_a = V^* - Q(a)$$
- Regret is a function of gaps and counts:

$$L_t = \mathbb{E}\left[\sum_{\tau=1}^{t} V^* - Q(a_\tau)\right]$$
$$= \sum_{a \in \mathcal{A}} \mathbb{E}[N_t(a)](V^* - Q(a))$$
$$= \sum_{a \in \mathcal{A}} \mathbb{E}[N_t(a)]\, \Delta_a$$

- A good algorithm ensures small counts for large gaps
- **Problem: gaps are not known!**

#### Simple Explanation

Total regret decomposes neatly: for each arm $a$, multiply **how often you pulled it** by **how suboptimal it was**, then sum across all arms.

$$L_t = \sum_{a} \underbrace{\mathbb{E}[N_t(a)]}_{\text{how often}} \times \underbrace{\Delta_a}_{\text{how bad}}$$

- A large gap $\Delta_a$ means arm $a$ is much worse than optimal — you want to pull it rarely.
- A small gap means the arm is nearly as good as optimal — pulling it occasionally is not costly.

The ideal algorithm quickly identifies bad arms and stops pulling them. The catch: you never directly observe $\Delta_a$, so you have to *infer* which arms are bad from noisy reward samples.

### Linear or Sublinear Regret

<img src=imgs/image-125.png width=500px />

- If an algorithm **forever** explores it will have linear total regret
- If an algorithm **never** explores it will have linear total regret
- Is it possible to achieve sublinear total regret?

#### Simple Explanation

The graph shows total regret over time for three strategies:

- **Greedy** (blue): never explores — locks in on whatever looked best early on, which may be suboptimal. Regret grows linearly because it keeps making the same mistake forever.
- **ε-greedy** (red): always explores at fixed rate — better than pure greedy, but still linear because it keeps wasting a fixed fraction of steps on random actions even after learning enough.
- **Decaying ε-greedy** (black): explores a lot early, then less and less over time — regret curve bends and flattens, achieving **sublinear** growth.

**Key insight**: both extremes (always explore, never explore) are equally bad in the long run — linear regret. The goal is an algorithm that explores *just enough* early on and gradually commits to the best arm, so that total regret grows sublinearly (e.g. $O(\log t)$).

### Greedy Algorithm

- We consider algorithms that estimate $\hat{Q}_t(a) \approx Q(a)$
- Estimate the value of each action by Monte-Carlo evaluation:

$$\hat{Q}_t(a) = \frac{1}{N_t(a)} \sum_{t=1}^{T} r_t \mathbf{1}(a_t = a)$$

- The **greedy** algorithm selects the action with highest estimated value:

$$a_t^* = \underset{a \in \mathcal{A}}{\operatorname{argmax}}\, \hat{Q}_t(a)$$

- Greedy can lock onto a suboptimal action forever
- $\Rightarrow$ Greedy has linear total regret

#### Simple Explanation

- **$\hat{Q}_t(a)$**: running average of all rewards received when arm $a$ was pulled. The indicator $\mathbf{1}(a_t = a)$ just means "only count timesteps where we actually chose $a$".
- **Greedy selection**: always pull the arm with the highest average reward so far — pure exploitation, zero exploration.

**Why it fails**: if by chance a suboptimal arm looks good early on (lucky initial pulls), greedy commits to it and never tries the others enough to discover the true best arm. That early mistake compounds forever → linear regret.